# ASSIGNMENT 1 INSTRUCTIONS

**This assignment accomplishes the following objectives:**
1. Learn how to invoke the OpenAI and Groq APIs
2. Interpret usage statistics (tokens and latency)
3. Recognize the limitations of standalone LLMs
4. Understand the difference between an LLM and an application built on top of it (like ChatGPT)
5. Compare inference performance across providers

**In order to complete the assignment:**
1. Section 1: Assignment setup: read through and pay attention to the code and comments as you work through each cell.
2. Section 2: Briefly answer all eight questions (Q1-Q8) inline as you work through each cell. Please ponder these questions yourself first - based on the output you see in each cell. You are then permitted to use AI chatbots to check your answer and better understand the outputs.

**To submit:**
1. Rename the assignment as Assignment1-\<asurite\>
2. Runtime -> Restart session and run all
3. Ctrl-s
4. File -> Download .ipynb
5. Submit on Canvas


# SECTION 1: ASSIGNMENT SETUP (JUST READ THROUGH)

### We'll first import the required libraries.

In [1]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# install the orchestration layer (aisuite) plus the provider SDKs (OpenAI, Groq)
# in Colab, this is typically needed once per new runtime
!pip install aisuite openai groq

### Next, we'll setup the API KEYS and Clients.

In [2]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

import os
import aisuite
from openai import OpenAI
from groq import Groq

# load API KEYS (Google Colab + local env fallback)
try:
  # read the API KEYs from Colab Secrets and expose it as an env variable
  from google.colab import userdata
  os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
  os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
  # read the API KEYs from local env and expose it as an env variable
  os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY")
  os.environ["GROQ_API_KEY"] = os.environ.get("GROQ_API_KEY")

# create the clients
openai_client = OpenAI()
groq_client = Groq()
aisuite_client = aisuite.Client()

# select the models you want to use
OPENAI_MODEL= "gpt-4.1-mini"
AISUITE_OPENAI_MODEL = "openai:"+OPENAI_MODEL
GROQ_MODEL = "llama-3.3-70b-versatile"
AISUITE_GROQ_MODEL = "groq:"+GROQ_MODEL

### The *run_openai_query* function accepts the system and user prompts from the user and invokes the OpenAI API through aisuite. It then prints the response from OpenAI as well as the usage stats (input/prompt tokens, output/completion tokens, and elapsed time).

In [3]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# used for wall-clock timing for request
import time

def run_openai_query(system_prompt, user_prompt):
  """
    Execute the query, print the response, and detailed usage stats.
  """

  # start the time before the API call
  start_time = time.time()

  # make chat-completions request thro aisuite, routed to OpenAI-backed model
  completion = aisuite_client.chat.completions.create(
      model=AISUITE_OPENAI_MODEL, # OpenAI-backed model
      messages=[
          {"role": "system", "content": system_prompt}, # system sets global behavior/instructions
           {"role": "user", "content": user_prompt} # user contains the actual task/prompt
      ],
  )

  # compute end-to-end wall-clock time for API request
  elapsed_time = time.time() - start_time

  # display response
  response = completion.choices[0].message.content.strip()
  print(f"\n--- RESPONSE ---")
  print(response)

  # display usage stats
  usage = completion.usage
  print(f"\n--- USAGE STATS ---")
  print(f"• Prompt Tokens:  {usage.prompt_tokens}")
  print(f"• Completion Tokens: {usage.completion_tokens}")
  print(f"• Total Tokens:  {usage.total_tokens}")
  if hasattr(usage, 'total_time'):
    print(f"• Total Time:  {usage.total_time:.2f}s")
  else:
    print(f"• Total Time:  {elapsed_time:.2f}s")


### The *run_groq_query* function accepts the system and user prompts from the user and invokes the Groq API through aisuite. It then prints the response from Groq as well as the usage stats (input/prompt tokens, output/completion tokens, and elapsed time).

In [4]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# used for wall-clock timing for request
import time

def run_groq_query(system_prompt, user_prompt):
  """
    Execute the query, print the response, and detailed usage stats.
  """

  # start the time before the API call
  start_time = time.time()

  # make chat-completions request thro aisuite, routed to Groq-backend model
  completion = aisuite_client.chat.completions.create(
      model=AISUITE_GROQ_MODEL, # Groq-backend model
      messages=[
          {"role": "system", "content": system_prompt}, # system sets global behavior/instructions
           {"role": "user", "content": user_prompt} # user contains the actual task/prompt
      ],
  )

  # compute end-to-end wall-clock time for API request
  elapsed_time = time.time() - start_time

  # display response
  response = completion.choices[0].message.content.strip()
  print(f"\n--- RESPONSE ---")
  print(response)

  # display usage stats
  usage = completion.usage
  print(f"\n--- USAGE STATS ---")
  print(f"• Prompt Tokens:  {usage.prompt_tokens}")
  print(f"• Completion Tokens: {usage.completion_tokens}")
  print(f"• Total Tokens:  {usage.total_tokens}")
  if hasattr(usage, 'total_time'):
    print(f"• Total Time:  {usage.total_time:.2f}s")
  else:
    print(f"• Total Time:  {elapsed_time:.2f}s")


# SECTION 2: LEARN THE LIMITATIONS OF BASIC LLMs, AND COMPARE OPENAI's and GROQ's PERFORMANCE.

### Run this cell to say Hello to OpenAI's LLM!<br>
**Notes**:  
- LLMs do not process text directly. Instead, text is converted into **tokens** - small units that may represent whole words, parts of words, numbers, or punctuation. For example, the word *"tokenization"* might be split into multiple tokens. Because **each model family uses its own tokenizer**, the same text can result in different token counts across models.
- There is a (one-time) cost to **train** LLM models, and there is a recurring variable cost incurred everytime you use LLM models (called **inference**). This is why LLM providers typically have usage or consumption-based pricing for API calls based on input and output tokens. For instance, here's what OpenAI's and Groq's pricing look like: https://openai.com/api/pricing/; https://groq.com/pricing.<br>

In [5]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# Our first OpenAI API call - say hello!
system_prompt = (
    "Provide a high-density, informative response. Avoid filler. "
    "Do not be overly brief, but do not be verbose. Aim for 'the perfect spot': "
    "comprehensive enough to answer fully, but concise enough to read quickly."
)
user_prompt = (
    "Hello!"
)
run_openai_query(system_prompt, user_prompt)


--- RESPONSE ---
Hello! How can I assist you today?

--- USAGE STATS ---
• Prompt Tokens:  56
• Completion Tokens: 9
• Total Tokens:  65
• Total Time:  1.04s


### Check the knowledge cutoff date for OpenAI's LLM. Then Answer Q1 and Q2 inline. <br>**Note**: we're making a distinction between OpenAI's LLM (that we're accessing through the API in this assignment) and OpenAI's ChatGPT Chatbot (that we've all used in the past).

In [6]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# Let's check the knowledge cutoff date on this model:
system_prompt = (
    "Provide a high-density, informative response. Avoid filler. "
    "Do not be overly brief, but do not be verbose. Aim for 'the perfect spot': "
    "comprehensive enough to answer fully, but concise enough to read quickly."
)
user_prompt = (
    "When were you last updated?"
)
run_openai_query(system_prompt, user_prompt)


--- RESPONSE ---
My knowledge was last updated in June 2024. I do not have information on events or developments occurring after that date.

--- USAGE STATS ---
• Prompt Tokens:  60
• Completion Tokens: 25
• Total Tokens:  85
• Total Time:  1.64s


In [7]:
##############################################################
# Based on the above cell's output, answer the following questions:
#  Q1) What response did you get for the knowledge cutoff date?: "My training includes information up until June 2024. I do not have real-time update capabilities beyond that point."
#
#  Q2) If the date is in the past, how is the ChatGPT app able to
#      provide current information?: ChatGPT can provide current information through Retrieval-Augmented Generation (RAG), which retrieves up-to-date information from external sources and uses it in the response.
#
##############################################################

### Ask OpenAI's LLM the current time. Then answer Q3 and Q4 inline. <br>**Note**: we're making a distinction between OpenAI's LLM (that we're accessing through the API in this assignment) and OpenAI's ChatGPT Chatbot (that we've all used in the past).

In [8]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# Ask for the current time:
system_prompt = (
    "Provide a high-density, informative response. Avoid filler. "
    "Do not be overly brief, but do not be verbose. Aim for 'the perfect spot': "
    "comprehensive enough to answer fully, but concise enough to read quickly."
)
user_prompt = (
    "What is the current time?"
)
run_openai_query(system_prompt, user_prompt)


--- RESPONSE ---
I don’t have real-time capabilities to access the current time. Please check your device’s clock or use an online time service for the accurate current time.

--- USAGE STATS ---
• Prompt Tokens:  60
• Completion Tokens: 31
• Total Tokens:  91
• Total Time:  1.61s


In [9]:
##############################################################
# Based on above cell's output, answer the following questions:
#  Q3) What response did you get?: "I don’t have access to real-time data, including the current time. Please check your device’s clock or use an online time service."
#
#  Q4) Why do you think we're getting the answer we are?: Why do you think we're getting the answer we are?: We are getting this answer because the OpenAI LLM used through the API in this assignment does not have access to real-time data or tool-based retrieval, so it cannot check the current time.
#
##############################################################

### Ask OpenAI's LLM about a recent event. Then answer Q5 and Q6 inline. <br>**Note**: we're making a distinction between OpenAI's LLM (that we're accessing through the API in this assignment) and OpenAI's ChatGPT Chatbot (that we've all used in the past).

In [10]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# Let's ask it about a current event:
system_prompt = (
    "Provide a high-density, informative response. Avoid filler. "
    "Do not be overly brief, but do not be verbose. Aim for 'the perfect spot': "
    "comprehensive enough to answer fully, but concise enough to read quickly."
)
user_prompt = (
    "Tell me about the Netflix Warner Bros acquisition."
)
run_openai_query(system_prompt, user_prompt)


--- RESPONSE ---
There has been no official acquisition of Warner Bros by Netflix. However, there have been significant collaborations and negotiations between the two companies in recent years, particularly concerning content licensing and streaming rights.

**Key context:**

- Warner Bros. is a major film and TV studio owned by Warner Bros. Discovery, a media conglomerate created through the merger of WarnerMedia and Discovery, Inc. in April 2022.
- Netflix is a leading global streaming platform, heavily investing in original content and acquiring streaming rights to bolster its library.
- Warner Bros. produces large franchises (e.g., DC Comics properties, Harry Potter, etc.) which are highly valuable for streaming platforms.

**Past developments:**

- Netflix and Warner Bros. have had content licensing deals, but many Warner Bros. properties have moved to HBO Max (now rebranded as Max), Warner’s own streaming service.
- There has been speculation and industry talk about potential m

In [11]:
##############################################################
# Based on above cell's output, answer the following questions:
#  Q5) What response did you get?: "As of mid-2024, there has been no confirmed acquisition of Warner Bros by Netflix. Both companies remain independent, with Warner Bros under Warner Bros. Discovery and Netflix operating as a standalone streaming giant.
# However, there has been speculation and industry discussion around possible collaborations, content licensing, or strategic partnerships between Netflix and Warner Bros. Discovery, but no formal acquisition deal has been announced or completed. Warner Bros content appears on multiple platforms, including HBO Max (now Max) which is Warner Bros. Discovery’s in-house streaming service.
# If you are referring to any recent news or rumors after June 2024, please provide more details or check the latest reliable sources for updates."
#
#  Q6) If the LLM didn't have the correct answer, how is the
#      ChatGPT app able to provide current information?: The information from the LLM is not up to date, because the LLM is only provided information from the training data. ChatGPT using RAG, can access up to date information from the web.
#
##############################################################

### Run the following cell to compare OpenAI to Groq. Then answer Q7 and Q8 inline. <br>
**Notes**:  
- LLMs do not process text directly. Instead, text is converted into **tokens** - small units that may represent whole words, parts of words, numbers, or punctuation. For example, the word *"tokenization"* might be split into multiple tokens. Because **each model family uses its own tokenizer**, the same text can result in different token counts across models.  
- There is a (one-time) cost to **train** LLM models, and there is a recurring variable cost incurred everytime you use LLM models (called **inference**). This is why LLM providers typically have usage or consumption-based pricing for API calls based on input and output tokens. For instance, here's what OpenAI's and Groq's pricing look like: https://openai.com/api/pricing/; https://groq.com/pricing.<br>
- This is where **Groq** comes in: Groq is an AI infrastructure company that developed the Language Processing Unit (LPU), a specialized chip designed specifically for high-speed, low-latency inference of LLMs. Unlike traditional GPUs, Groq's hardware enables near-instantaneous, real-time responses from AI models like Llama and Mistral.



In [12]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# Let's compare OpenAI to Groq on a task:
system_prompt = (
    "Provide a high-density, informative response. Avoid filler. "
    "Do not be overly brief, but do not be verbose. Aim for 'the perfect spot': "
    "comprehensive enough to answer fully, but concise enough to read quickly."
)
user_prompt = (
    "Output the integers from 1 to 1500, in a comma-separated list. "
    "Do not include any extra spaces or text."
)

print ("\n############ OpenAI Query ############")
run_openai_query(system_prompt, user_prompt)

print ("\n############ Groq Query ############")
run_groq_query(system_prompt, user_prompt)


############ OpenAI Query ############

--- RESPONSE ---
1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,26

In [13]:
##############################################################
# Based on above cell's output, answer the following questions:
#  Q7) Why did we pick this task to compare OpenAI to Groq?: Using this task we are able to assess how efficient OpenAI is to Groq by making it perform a simple task, and providing stats to for comparison.
#
#  Q8) Can you explain the difference in usage statistics?: The difference occurs because OpenAI and Groq has their own tokenizers, splitting the text into tokens differently. Groq also has LPU, which is built for speed.
#
##############################################################